# 2.4 GHz slotlu patch anten: az veriyle S11 ve rezonans frekansı tahmini
Her kod hücresi **bağımsızdır**: kendi importlarını yapar ve ara sonuçları `artifacts/` klasöründen okur. Bir hücreyi tekrar çalıştırmak için öncekileri yeniden koşmana gerek yok (ilk kez çalıştırırken sırayla koş).

Veri: Zenodo `DataSet.csv` (https://zenodo.org/records/17527112). Satırlar frekans noktalarıdır; gerçek örnek sayısı farklı geometri sayısıdır.

In [ ]:
# HÜCRE 1: ayarlar (tek yer; diğer hücreler bunu kendi içinde tekrar tanımlar)
from pathlib import Path
Path('artifacts').mkdir(exist_ok=True)
print('hazır')

## 2. Veriyi yükle ve temizle

In [ ]:
import pandas as pd
DATA_PATH = 'DataSet.csv'   # dosya yolunu gerekirse değiştir
OUT = 'artifacts'
df = pd.read_csv(DATA_PATH, sep=';')
df.columns = ['w_sub','w_line','l_line','h_sub','h_patch','w_slot','l_slot','w_patch','l_sub','l_patch','f','s11']
print(df.shape, 'eksik:', int(df.isna().sum().sum()), 'yinelenen:', int(df.duplicated().sum()))
df.to_csv(f'{OUT}/rows.csv', index=False)
df.head()

## 3. Geometri tablosu (53 gerçek örnek) ve hedefler

In [ ]:
import pandas as pd
OUT = 'artifacts'
df = pd.read_csv(f'{OUT}/rows.csv')
FEATS = ['w_sub','l_line','w_slot','l_slot','w_patch','l_sub','l_patch']  # sabit olanlar (w_line, h_sub, h_patch) dışarıda
def targets(x):
    i = x.s11.idxmin()
    return pd.Series({'f_res': x.loc[i,'f'], 's11_min': x.s11.min()})
geo = df.groupby(FEATS).apply(targets).reset_index()
geo['geo_id'] = range(len(geo))
geo['basarili'] = (geo.s11_min <= -10).astype(int)
geo.to_csv(f'{OUT}/geo.csv', index=False)
print(len(geo), 'geometri;', 'S11_min <= -10 dB oranı:', round(geo.basarili.mean(),2))
geo.describe().T[['mean','std','min','max']]

## 4. Veriyi tanı: örnek S11 eğrileri ve hedef dağılımları

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
OUT = 'artifacts'
df = pd.read_csv(f'{OUT}/rows.csv'); geo = pd.read_csv(f'{OUT}/geo.csv')
FEATS = ['w_sub','l_line','w_slot','l_slot','w_patch','l_sub','l_patch']
fig, ax = plt.subplots(1, 3, figsize=(15,4))
for _, r in geo.sample(6, random_state=0).iterrows():
    m = (df[FEATS] == r[FEATS]).all(axis=1)
    ax[0].plot(df.loc[m,'f'], df.loc[m,'s11'], lw=1)
ax[0].axhline(-10, color='k', ls='--', lw=0.8); ax[0].set(xlabel='Frekans (GHz)', ylabel='S11 (dB)', title='Örnek S11 eğrileri')
ax[1].hist(geo.f_res*1000, bins=15); ax[1].set(xlabel='f_res (MHz)', title='Rezonans frekansı dağılımı')
ax[2].hist(geo.s11_min, bins=15); ax[2].set(xlabel='S11_min (dB)', title='S11 çukur derinliği')
plt.tight_layout(); plt.savefig(f'{OUT}/eda.png', dpi=150); plt.show()

## 5. Geometri bazlı çapraz doğrulama: baz model, Ridge, Random Forest, GP
Veri zaten geometri başına tek satır olduğu için satır sızıntısı yok.

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from scipy.stats import spearmanr
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as Ck
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
OUT = 'artifacts'
FEATS = ['w_sub','l_line','w_slot','l_slot','w_patch','l_sub','l_patch']
geo = pd.read_csv(f'{OUT}/geo.csv'); X = geo[FEATS].values
def make_models():
    return {
      'baz (ortalama)': DummyRegressor(),
      'Ridge': make_pipeline(StandardScaler(), Ridge(1.0)),
      'Random Forest': RandomForestRegressor(300, random_state=0),
      'GP': make_pipeline(StandardScaler(), GaussianProcessRegressor(Ck(1.0)*RBF(np.ones(len(FEATS)))+WhiteKernel(0.1), normalize_y=True, n_restarts_optimizer=3, random_state=0)),
    }
rows = []
preds = {}
kf = KFold(5, shuffle=True, random_state=0)
for target, scale, unit in [('f_res', 1000, 'MHz'), ('s11_min', 1, 'dB')]:
    y = geo[target].values
    for name, m in make_models().items():
        p = cross_val_predict(m, X, y, cv=kf)
        e = (p - y) * scale
        rho = spearmanr(p, y)[0] if name != 'baz (ortalama)' else np.nan
        rows.append(dict(hedef=target, model=name, MAE=np.abs(e).mean(), RMSE=np.sqrt((e**2).mean()), birim=unit, spearman=rho))
        preds[f'{target}|{name}'] = p
res = pd.DataFrame(rows).round(2)
res.to_csv(f'{OUT}/cv_sonuc.csv', index=False)
pd.DataFrame(preds).assign(geo_id=geo.geo_id).to_csv(f'{OUT}/cv_tahmin.csv', index=False)
res

## 5b. Tekrarlı doğrulama (20 farklı rastgele bölme, ortalama ± standart sapma)
53 örnekte tek bölme oynak olabilir; bu yüzden 5 katlı doğrulamayı 20 farklı rastgele bölmeyle tekrarlayıp ortalama ve standart sapmayı raporla. Birkaç dakika sürebilir.

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as Ck
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
OUT = 'artifacts'
FEATS = ['w_sub','l_line','w_slot','l_slot','w_patch','l_sub','l_patch']
geo = pd.read_csv(f'{OUT}/geo.csv'); X = geo[FEATS].values
def models():
    return {'baz (ortalama)': DummyRegressor(),
      'Ridge': make_pipeline(StandardScaler(), Ridge(1.0)),
      'Random Forest': RandomForestRegressor(200, random_state=0, n_jobs=-1),
      'GP': make_pipeline(StandardScaler(), GaussianProcessRegressor(Ck(1.0)*RBF(np.ones(len(FEATS)))+WhiteKernel(0.1), normalize_y=True, n_restarts_optimizer=1, random_state=0))}
R = 20; rows = []
for rep in range(R):
    kf = KFold(5, shuffle=True, random_state=rep)
    for t, sc in [('f_res', 1000), ('s11_min', 1)]:
        y = geo[t].values
        for n, m in models().items():
            p = cross_val_predict(m, X, y, cv=kf)
            rows.append(dict(rep=rep, hedef=t, model=n, MAE=np.abs(p-y).mean()*sc, RMSE=np.sqrt(((p-y)**2).mean())*sc))
d = pd.DataFrame(rows); d.to_csv(f'{OUT}/cv_tekrar_ham.csv', index=False)
s = d.groupby(['hedef','model'])[['MAE','RMSE']].agg(['mean','std']).round(2)
s.columns = ['MAE_ort','MAE_std','RMSE_ort','RMSE_std']; s = s.reset_index()
s.to_csv(f'{OUT}/cv_tekrar.csv', index=False)
for t in ['f_res','s11_min']:
    a = d[(d.hedef==t)&(d.model=='baz (ortalama)')].set_index('rep').MAE
    g = d[(d.hedef==t)&(d.model=='GP')].set_index('rep').MAE
    print(t, 'GP, baz modele göre MAE azalması: ort %', round(((1-g/a).mean())*100,1), '| en az %', round(((1-g/a).min())*100,1))
s

## 6. Veri azaltma deneyi
Eğitim geometri sayısını azaltıp test hatasının nasıl değiştiğini ölç (her boyut için tekrarlı rastgele bölme).

In [ ]:
import numpy as np, pandas as pd, warnings, matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as Ck
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
OUT = 'artifacts'
FEATS = ['w_sub','l_line','w_slot','l_slot','w_patch','l_sub','l_patch']
geo = pd.read_csv(f'{OUT}/geo.csv'); X = geo[FEATS].values
def make_models():
    return {
      'baz (ortalama)': DummyRegressor(),
      'Ridge': make_pipeline(StandardScaler(), Ridge(1.0)),
      'Random Forest': RandomForestRegressor(200, random_state=0),
      'GP': make_pipeline(StandardScaler(), GaussianProcessRegressor(Ck(1.0)*RBF(np.ones(len(FEATS)))+WhiteKernel(0.1), normalize_y=True, random_state=0)),
    }
SIZES, REPS, N_TEST = [10, 20, 30, 40], 10, 10
rng = np.random.default_rng(0); rows = []
for target, scale in [('f_res', 1000), ('s11_min', 1)]:
    y = geo[target].values
    for n in SIZES:
        for rep in range(REPS):
            idx = rng.permutation(len(geo)); te, tr = idx[:N_TEST], idx[N_TEST:N_TEST+n]
            for name, m in make_models().items():
                m.fit(X[tr], y[tr]); mae = np.abs(m.predict(X[te]) - y[te]).mean()*scale
                rows.append(dict(hedef=target, n_egitim=n, model=name, MAE=mae))
res = pd.DataFrame(rows).groupby(['hedef','n_egitim','model']).MAE.mean().reset_index()
res.to_csv(f'{OUT}/veri_azaltma.csv', index=False)
fig, ax = plt.subplots(1, 2, figsize=(11,4))
for a, t, u in zip(ax, ['f_res','s11_min'], ['MHz','dB']):
    for name, g in res[res.hedef==t].groupby('model'): a.plot(g.n_egitim, g.MAE, marker='o', label=name)
    a.set(xlabel='Eğitim geometri sayısı', ylabel=f'MAE ({u})', title=t); a.legend()
plt.tight_layout(); plt.savefig(f'{OUT}/veri_azaltma.png', dpi=150); plt.show()

## 7. Aday geometri üretme ve sıralama (öneri)
Hedef: `f_res` 2.4 GHz'e yakın ve `S11_min ≤ −10 dB`. 7 boyutlu uzayda 53 nokta çok seyrek olduğu için adaylar **gözlenen geometrilerin küçük sapmalarıyla** üretilir; GP'nin belirsizliği (std) de raporlanır ve belirsizliği yüksek adaylar elenir. **Öneriler simülatörle doğrulanmadı.**

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as Ck
from sklearn.preprocessing import StandardScaler
OUT = 'artifacts'
FEATS = ['w_sub','l_line','w_slot','l_slot','w_patch','l_sub','l_patch']
geo = pd.read_csv(f'{OUT}/geo.csv'); X = geo[FEATS].values
sc = StandardScaler().fit(X); Xs = sc.transform(X)
def gp(): return GaussianProcessRegressor(Ck(1.0)*RBF(np.ones(len(FEATS)))+WhiteKernel(0.1), normalize_y=True, n_restarts_optimizer=3, random_state=0)
m_f = gp().fit(Xs, geo.f_res.values); m_s = gp().fit(Xs, geo.s11_min.values)
rng = np.random.default_rng(0)
SAPMA = 0.05                                   # parametre aralığının %5'i kadar sapma
lo, hi = X.min(0), X.max(0)
base = X[rng.integers(0, len(X), 20000)]
cand = np.clip(base + rng.normal(0, SAPMA, base.shape) * (hi - lo), lo, hi)
cs = sc.transform(cand)
f_mu, f_sd = m_f.predict(cs, return_std=True); s_mu, s_sd = m_s.predict(cs, return_std=True)
c = pd.DataFrame(cand, columns=FEATS)
c['f_res_tahmin'] = f_mu; c['f_res_std_MHz'] = f_sd*1000
c['s11_min_tahmin'] = s_mu; c['s11_min_std_dB'] = s_sd
c['skor'] = (c.f_res_tahmin - 2.4).abs()*1000 + np.clip(c.s11_min_tahmin + 10, 0, None)*5 + c.f_res_std_MHz
top = c[(c.f_res_std_MHz < 15)].sort_values('skor').head(10).round(3)
top.to_csv(f'{OUT}/oneriler.csv', index=False)
print('Tahmin çeşitliliği (std):', round(c.f_res_tahmin.std()*1000,1), 'MHz,', round(c.s11_min_tahmin.std(),1), 'dB')
top

## 8. İsteğe bağlı: tüm S11 eğrisini tahmin et (geometri + frekans girdisi)
Bölme **geometriye göre** yapılır; aksi halde komşu frekans noktaları sızıntı yaratır.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
OUT = 'artifacts'
FEATS = ['w_sub','l_line','w_slot','l_slot','w_patch','l_sub','l_patch']
df = pd.read_csv(f'{OUT}/rows.csv'); geo = pd.read_csv(f'{OUT}/geo.csv')
d = df.merge(geo[FEATS+['geo_id']], on=FEATS)
tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=0).split(d, groups=d.geo_id))
cols = FEATS + ['f']
rf = RandomForestRegressor(100, n_jobs=-1, random_state=0).fit(d.iloc[tr][cols], d.iloc[tr].s11)
p = rf.predict(d.iloc[te][cols])
print('Test RMSE (dB):', round(float(np.sqrt(np.mean((p - d.iloc[te].s11.values)**2))), 2))
g = d.iloc[te].geo_id.iloc[0]; s = d.iloc[te][d.iloc[te].geo_id == g]
plt.plot(s.f, s.s11, label='CST sonucu'); plt.plot(s.f, rf.predict(s[cols]), label='Model tahmini')
plt.xlabel('Frekans (GHz)'); plt.ylabel('S11 (dB)'); plt.legend(); plt.savefig(f'{OUT}/egri_ornek.png', dpi=150); plt.show()

## 9. S11_min için iyileştirme denemeleri
Oran özellikleri, log dönüşümü ve Matern çekirdek tek tek ve birlikte denenir (anten bazlı, 5 kat × 10 tekrar).

In [ ]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'          # joblib işçi süreçlerinde uyarıları sustur
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor as GPR
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel as C
from joblib import Parallel, delayed
OUT = 'artifacts'
FEATS = ['w_sub','l_line','w_slot','l_slot','w_patch','l_sub','l_patch']
geo = pd.read_csv(f'{OUT}/geo.csv'); ham = geo[FEATS].values
def gp_kur(d, tur='rbf'):
    k = Matern(np.ones(d), nu=2.5) if tur == 'matern' else RBF(np.ones(d))
    return GPR(C(1.0)*k + WhiteKernel(0.1), normalize_y=True, n_restarts_optimizer=3, random_state=0)
# S11_min için denenen iyileştirmeler (oran özellikleri, log dönüşümü, Matern çekirdek)
def oranli(a):
    return np.c_[a, a[:,6]/a[:,4], a[:,3]/a[:,2], a[:,0]/a[:,4], a[:,5]/a[:,6]]
VARYANT = {'V0 RBF (ilk sürüm)': (False, False, 'rbf'), 'V1 + oran özellikleri': (True, False, 'rbf'),
           'V2 log dönüşümü': (False, True, 'rbf'), 'V3 Matern': (False, False, 'matern'),
           'V4 log + Matern': (False, True, 'matern'), 'V5 oran + log': (True, True, 'rbf')}
def kat(tr, te, ad):
    oran, log, tur = VARYANT[ad]
    A = oranli(ham) if oran else ham
    sc = StandardScaler().fit(A[tr]); y = geo['s11_min'].values
    m = gp_kur(A.shape[1], tur).fit(sc.transform(A[tr]), np.log(-y[tr]) if log else y[tr])
    mu = m.predict(sc.transform(A[te]))
    return ad, np.abs((-np.exp(mu) if log else mu) - y[te]).tolist()
bol = list(RepeatedKFold(n_splits=5, n_repeats=10, random_state=42).split(ham))
sonuc = Parallel(n_jobs=-1)(delayed(kat)(tr, te, ad) for tr, te in bol for ad in VARYANT)
hata = {}
for ad, h in sonuc: hata.setdefault(ad, []).extend(h)
pd.Series({ad: np.mean(h) for ad, h in hata.items()}).round(2).rename('S11_min MAE (dB)')

## 10. Taze bölmelerle doğrulama ve belirsizlik kalibrasyonu
Seçimin aynı bölmelere aşırı uymadığını görmek için farklı tohumla (123) tekrar ölçülür. Ayrıca ±1σ ve ±1.96σ aralıklarının gerçeği ne kadar kapsadığı sayılır.

In [ ]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'          # joblib işçi süreçlerinde uyarıları sustur
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor as GPR
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel as C
from joblib import Parallel, delayed
OUT = 'artifacts'
FEATS = ['w_sub','l_line','w_slot','l_slot','w_patch','l_sub','l_patch']
geo = pd.read_csv(f'{OUT}/geo.csv'); ham = geo[FEATS].values
def gp_kur(d, tur='rbf'):
    k = Matern(np.ones(d), nu=2.5) if tur == 'matern' else RBF(np.ones(d))
    return GPR(C(1.0)*k + WhiteKernel(0.1), normalize_y=True, n_restarts_optimizer=3, random_state=0)
# Taze bölmelerle (tohum 123) doğrulama + belirsizlik kalibrasyonu: %68/%95 aralıkları gerçeği kapsıyor mu?
KONF = [('f_res','GP-RBF','rbf',False), ('f_res','GP-Matern','matern',False),
        ('s11_min','GP-RBF','rbf',False), ('s11_min','GP-Matern+log','matern',True)]
def kat(tr, te):
    sc = StandardScaler().fit(ham[tr]); o = []
    for hedef, ad, tur, log in KONF:
        y = geo[hedef].values
        m = gp_kur(7, tur).fit(sc.transform(ham[tr]), np.log(-y[tr]) if log else y[tr])
        mu, sd = m.predict(sc.transform(ham[te]), return_std=True)
        if log: tah, lo, hi, lo1, hi1 = -np.exp(mu), -np.exp(mu+1.96*sd), -np.exp(mu-1.96*sd), -np.exp(mu+sd), -np.exp(mu-sd)
        else:   tah, lo, hi, lo1, hi1 = mu, mu-1.96*sd, mu+1.96*sd, mu-sd, mu+sd
        o.append((hedef, ad, np.abs(tah-y[te]), np.abs(y[tr].mean()-y[te]), (y[te]>=lo)&(y[te]<=hi), (y[te]>=lo1)&(y[te]<=hi1), (hi-lo)/2))
    return o
bol = list(RepeatedKFold(n_splits=5, n_repeats=20, random_state=123).split(ham))
res = Parallel(n_jobs=-1)(delayed(kat)(tr, te) for tr, te in bol)
satirlar = []
for j, (hedef, ad, _, _) in enumerate(KONF):
    h = np.concatenate([r[j][2] for r in res]); b = np.concatenate([r[j][3] for r in res])
    k95 = np.concatenate([r[j][4] for r in res]); k68 = np.concatenate([r[j][5] for r in res]); gen = np.concatenate([r[j][6] for r in res])
    satirlar.append(dict(hedef=hedef, model=ad, MAE=h.mean(), baz=b.mean(), iyilesme_yuzde=100*(1-h.mean()/b.mean()),
                         kapsama68=k68.mean(), kapsama95=k95.mean(), sd_hata_korelasyon=np.corrcoef(gen, h)[0,1]))
pd.DataFrame(satirlar).round(3)

## 11. Veriden uzak geometriler testi
Model, eğitim verisinden uzak antenlerde hatasını büyütürken belirsizliğini de büyütüyor mu?

In [ ]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'          # joblib işçi süreçlerinde uyarıları sustur
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor as GPR
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel as C
from joblib import Parallel, delayed
OUT = 'artifacts'
FEATS = ['w_sub','l_line','w_slot','l_slot','w_patch','l_sub','l_patch']
geo = pd.read_csv(f'{OUT}/geo.csv'); ham = geo[FEATS].values
def gp_kur(d, tur='rbf'):
    k = Matern(np.ones(d), nu=2.5) if tur == 'matern' else RBF(np.ones(d))
    return GPR(C(1.0)*k + WhiteKernel(0.1), normalize_y=True, n_restarts_optimizer=3, random_state=0)
# Veriden uzak geometriler: en uzak 10 anten teste, kalanı eğitime (karşılaştırma: rastgele 10 anten, 20 tekrar)
sc0 = StandardScaler().fit(ham); Z = sc0.transform(ham)
D = np.linalg.norm(Z[:,None]-Z[None], axis=2); np.fill_diagonal(D, np.inf); nn = D.min(1)
uzak = np.argsort(-nn)[:10]
def test(tr, te):
    sc = StandardScaler().fit(ham[tr]); o = {}
    for hedef, log in (('f_res', False), ('s11_min', True)):
        y = geo[hedef].values
        m = gp_kur(7, 'matern').fit(sc.transform(ham[tr]), np.log(-y[tr]) if log else y[tr])
        mu, sd = m.predict(sc.transform(ham[te]), return_std=True)
        tah = -np.exp(mu) if log else mu
        o[hedef] = (np.abs(tah-y[te]).mean(), (np.exp(mu)*sd if log else sd).mean())
    return o
u = test(np.setdiff1d(np.arange(len(geo)), uzak), uzak)
rng = np.random.default_rng(0); r = []
for _ in range(20):
    te = rng.choice(len(geo), 10, replace=False); r.append(test(np.setdiff1d(np.arange(len(geo)), te), te))
pd.DataFrame({'rastgele 10': {h: np.mean([x[h][0] for x in r]) for h in ('f_res','s11_min')},
              'en uzak 10': {h: u[h][0] for h in ('f_res','s11_min')}}).T.round(3)   # f_res GHz, s11_min dB

## 12. Tüm S11 eğrisini tahmin etme (uygulamadaki eğri grafiği)
Eğriler PCA ile 8 sayıya indirilir, her sayı için bir GP kurulur.

In [ ]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'          # joblib işçi süreçlerinde uyarıları sustur
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor as GPR
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel as C
from joblib import Parallel, delayed
OUT = 'artifacts'
FEATS = ['w_sub','l_line','w_slot','l_slot','w_patch','l_sub','l_patch']
geo = pd.read_csv(f'{OUT}/geo.csv'); ham = geo[FEATS].values
def gp_kur(d, tur='rbf'):
    k = Matern(np.ones(d), nu=2.5) if tur == 'matern' else RBF(np.ones(d))
    return GPR(C(1.0)*k + WhiteKernel(0.1), normalize_y=True, n_restarts_optimizer=3, random_state=0)
# Tüm S11 eğrisini tahmin etme (PCA + Gaussian Process): çukur yeri ve derinliği eğriden türetilir
from sklearn.decomposition import PCA
rows = pd.read_csv(f'{OUT}/rows.csv')
F = np.linspace(1.8, 2.8, 201)
E = []
for _, r in geo.iterrows():
    m = (rows[FEATS] == r[FEATS]).all(axis=1); d = rows[m].sort_values('f')
    E.append(np.interp(F, d.f.values, d.s11.values))
E = np.array(E)
def kat(tr, te, k=8):
    sc = StandardScaler().fit(ham[tr]); ort = E[tr].mean(0)
    pca = PCA(k).fit(E[tr] - ort); Ztr = pca.transform(E[tr] - ort)
    zt = np.column_stack([gp_kur(7, 'matern').fit(sc.transform(ham[tr]), Ztr[:, j]).predict(sc.transform(ham[te])) for j in range(k)])
    P = pca.inverse_transform(zt) + ort
    return (np.abs(F[P.argmin(1)] - geo.f_res.values[te]).tolist(), np.abs(P.min(1) - geo.s11_min.values[te]).tolist())
bol = list(RepeatedKFold(n_splits=5, n_repeats=10, random_state=123).split(ham))
res = Parallel(n_jobs=-1)(delayed(kat)(tr, te) for tr, te in bol)
f = np.concatenate([r[0] for r in res]); s = np.concatenate([r[1] for r in res])
print(f'Eğri modeli: f_res MAE = {f.mean()*1000:.1f} MHz, S11_min MAE = {s.mean():.2f} dB')